# NEXUS LLM Extension -- LLM vs. Hand-Written Comparison

Builds on `llm_skill_generation_training.ipynb`. 

1. Trains the hand-written policy and an LLM-generated policy on the same
   config/env/seeds and compares `returns/env_reward_mean` 
   from `scripts.run_llm_comparison.compare_policies`.
2. Runs the same comparison across the environment suite and plots it.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

BACKEND = "mock"   # "mock" | "hf" | "openai"
SEEDS = 3

## 1. Single environment comparison

In [ ]:
from nexus_continuous.envs.env_registry import ENV_REGISTRY
from nexus_continuous.llm.client import LLMClient, LLMConfig, MockSkillGenerator
from nexus_continuous.scripts.run_llm_comparison import compare_policies

ENV_CONFIGS = {
    "CartpoleBalance": "configs/cartpole_balance_nesy.yaml",
    "CheetahRun": "configs/cheetah_run_nesy.yaml",
}

def make_client(env_name, seed=0):
    if BACKEND != "mock":
        return LLMClient(LLMConfig(backend=BACKEND))
    fields = ENV_REGISTRY[env_name]["fields"]
    return LLMClient(
        LLMConfig(backend="mock", seed=seed),
        mock_generator=MockSkillGenerator(fields, seed=seed)
    )

env_name = "CartpoleBalance"
result = compare_policies(
    env_name, 
    ENV_CONFIGS[env_name], 
    num_seeds=SEEDS,
    client=make_client(env_name)
)
print("Handwritten:", result["handwritten"])
print("LLM:        ", result["llm"])

In [ ]:
from nexus_continuous.llm.plot import plot_comparison

os.makedirs("plots", exist_ok=True)
path = plot_comparison(result["handwritten"], result["llm"], env_name, f"plots/{env_name}_comparison.png")
plt.figure(figsize=(4.5, 4)); plt.imshow(plt.imread(path)); plt.axis("off"); plt.show()

## 2. Full suite comparison

In [ ]:
rows = []
for env_name, cfg_path in ENV_CONFIGS.items():
    row = compare_policies(env_name, cfg_path, num_seeds=SEEDS, client=make_client(env_name))
    rows.append({k: row[k] for k in ("env", "hand_mean", "hand_std", "llm_mean", "llm_std")})
    print(env_name, "done")

df = pd.DataFrame(rows)
df

In [ ]:
x = np.arange(len(df)); width = 0.35
plt.figure(figsize=(9, 4))
plt.bar(x - width / 2, df.hand_mean, width, yerr=df.hand_std, label="Handwritten", capsize=4)
plt.bar(x + width / 2, df.llm_mean, width, yerr=df.llm_std, label="LLM", capsize=4)
plt.xticks(x, df.env, rotation=20)
plt.ylabel("Mean env reward (over seeds)")
plt.legend(); plt.tight_layout(); plt.show()

os.makedirs("results", exist_ok=True)
df.to_csv("results/llm_vs_handwritten_comparison.csv", index=False)